This notebook contains Python code for reproducing the results in our paper on using a large language model to give feedback on student answers to open-ended questions:

Van Campenhout, R., Dittel, J. S., Jerome, B., Clark, M. W., & Johnson, B. G. (2025). Open-ended questions need personalized feedback: Analyzing LLM-enabled features with student data. In *Proceedings of the Second Workshop on Automated Evaluation of Learning and Assessment Content at the 26th International Conference on Artificial Intelligence in Education (AIED 2025)*. [https://drive.google.com/file/d/15HCyN1uU6AtIT8aVpMJCS5ZQa75buA_g/view](https://drive.google.com/file/d/15HCyN1uU6AtIT8aVpMJCS5ZQa75buA_g/view)

This paper was presented at [AIED 2025](https://aied2025.itd.cnr.it/) as part of the [Second Workshop on Automated Evaluation of Learning and Assessment Content](https://sites.google.com/cam.ac.uk/eval-lac-2025).

Results are presented in the order they occur, organized by the paper's sections. For each result, an excerpt from the paper is given followed by code to compute the result from the data set provided. Example:

>This resulted in a dataset of 83,624 LLM-enabled question sessions (56,944 exam question and 26,680 C&C)...

```
sessions = pd.concat( [ exam_sessions, cc_sessions ] )
print( f'{len( sessions )} sessions' )
print( f'{len( exam_sessions )} exam question sessions' )
print( f'{len( cc_sessions )} C&C sessions' )
```

Please refer to the paper for additional context.

In [1]:
import difflib
import re

import pandas as pd
from scipy.stats import chi2_contingency, mannwhitneyu

## Read dataset

Open-ended questions.

In [2]:
exam_events = pd.read_parquet( 'exam_events.parquet' )
exam_sessions = pd.read_parquet( 'exam_sessions.parquet' )
cc_events = pd.read_parquet( 'cc_events.parquet' )
cc_sessions = pd.read_parquet( 'cc_sessions.parquet' )

In [3]:
exam_events.head()

,timestamp,student_id,question_id,textbook_id,subject,question,attempt_number,answer,feedback,attempt_category
1,2024-09-03 18:18:54,ZCD7MS8ETNX5HC2JU782,c88647b63133d091d87baa3b0d57cd23bfde104ab4c8d2...,9781071845226,Social Science,"Write an exam question for the section ""Femini...",1,Describe how the historical and social context...,Your question is well-formulated and captures ...,+
2,2024-09-03 18:26:28,XMUPT74AZHNN2A37XGFU,e12344b7f4eb9c6ecc5a02e109c7375fead99c751f9e94...,9781544349848,Language Arts & Disciplines,"Write an exam question for the section ""Commun...",1,Where in your manual can you find the learning...,"OK, no problem. A good example of an exam qu...",x
3,2024-09-03 18:30:42,XMUPT74AZHNN2A37XGFU,9bf3f7ffe128c3c985990b7ebacf0fdf41863e89483ecc...,9781544349848,Language Arts & Disciplines,"Write an exam question for the section ""Commun...",1,Which communication style does this identify.....,Your question does a good job of capturing the...,+
4,2024-09-03 18:46:12,QEVKMPHTAMVUGMDFKEYU,e12344b7f4eb9c6ecc5a02e109c7375fead99c751f9e94...,9781544349848,Language Arts & Disciplines,"Write an exam question for the section ""Commun...",1,Describe the specific cultural expectations th...,Your question does a good job of capturing sev...,+
5,2024-09-03 18:56:46,GWS3JNKQPPVBKBE5C4T2,dce3e0093f64caa188cdb3bac64ca9bc62d176687f04d1...,9781492594192,Medical,"Write a test question about this section, ""Dru...",1,Do all cells have the same receptors?,"Your question, ""Do all cells have the same rec...",+


In [4]:
exam_sessions.head()

,student_id,question_id,textbook_id,subject,assigned,pattern,first_attempt,second_attempt,second_attempt_elapsed,second_attempt_overlap,rating
0,5WHTDMVRDZQCMMJSQSQ8,0001890c3830597db6f5ec846ab48bc067f51922b5aaad...,9781000164985,Language Arts & Disciplines,0,+,+,None,NaN,NaN,None
1,RVQX22VRYWFDCXQ2ZX2N,0001890c3830597db6f5ec846ab48bc067f51922b5aaad...,9781000164985,Language Arts & Disciplines,0,+,+,None,NaN,NaN,None
2,EQ465MRKHNJFYR7UAQFG,0001b5af3b6b1a8fa223c7b9ce4a04289d7421de4f6e41...,9781774625644VS,None,0,+,+,None,NaN,NaN,None
3,7RKR8PX86W62NSGVPVMA,00038b47d8b9c3dc657b6e5b887f2fefcf4fa98c6a5d65...,9781351857345,Language Arts & Disciplines,0,+,+,None,NaN,NaN,None
4,PQZSS7YURKUJCXZPYZ7V,0005b2ab2d3586f63b9458c3f71744677ac80d08e456fb...,9781000996906,Family & Relationships,0,+,+,None,NaN,NaN,None


In [5]:
cc_events.head()

,timestamp,student_id,question_id,textbook_id,subject,question,attempt_number,answer,feedback,attempt_category
0,2024-09-03 18:39:45,UQ7SXTD4M572WXPTQY6A,d589e4d930b616166dfb179251d681e16dfb66e3f45fd0...,9781492591078,Health & Fitness,Explain the difference between the terms lacta...,1,ggg\n,"OK, no problem. The lactate threshold refers t...",x
1,2024-09-03 18:48:09,RPTXVUMDPYR8PA4TCNVM,c00a6cd9bce1b837e8ac432235e439b7f76e9d63ee1593...,9781071845226,Social Science,Explain the difference between the term Acquai...,1,"In contrast to acquaintance rape, which concen...",Your explanation is accurate and well-articula...,+
2,2024-09-03 18:50:46,RPTXVUMDPYR8PA4TCNVM,cc4e22524ceef62cdb66f969a73772ae4e032403617791...,9781071845226,Social Science,Explain the difference between the term Incapa...,1,A victim of spousal rape is one who is married...,Your explanation captures the primary distinct...,+
3,2024-09-03 18:55:10,RPTXVUMDPYR8PA4TCNVM,c81b35f4cb1c9d94c456723608241a930ca1204550fdee...,9781071845226,Social Science,Explain the difference between the term Spousa...,1,"The absence of permission inside a marriage, b...",Your explanation captures the key differences ...,+
4,2024-09-03 18:59:39,RPTXVUMDPYR8PA4TCNVM,b9233fbbd3340ede1c151675c3b38489e3ac2331a9f611...,9781071845226,Social Science,Explain the difference between the term Nation...,1,The NISVS specializes in intimate partner and ...,Your explanation is accurate and well-stated. ...,+


In [6]:
cc_sessions.head()

,student_id,question_id,textbook_id,subject,assigned,pattern,first_attempt,second_attempt,second_attempt_elapsed,second_attempt_overlap,rating
0,3QGXYCPC5JFSV8XN6J53,000eecaf7843c60c47ff3f41bff918991d4f7a54653160...,9781071833872,Social Science,0,-,-,None,NaN,NaN,None
1,6ABHA6Y4EMMHEMAN2UVF,000eecaf7843c60c47ff3f41bff918991d4f7a54653160...,9781071833872,Social Science,0,+,+,None,NaN,NaN,None
2,7RMQKUNKBNDAY7X6T5SH,000eecaf7843c60c47ff3f41bff918991d4f7a54653160...,9781071833872,Social Science,0,-+,-,+,73.0,0.859649,None
3,UAGUW2YZA2XRKRXNYTWM,000eecaf7843c60c47ff3f41bff918991d4f7a54653160...,9781071833872,Social Science,0,x,x,None,NaN,NaN,None
4,WT5QRZJPBTUMH5CBANJY,000eecaf7843c60c47ff3f41bff918991d4f7a54653160...,9781071833872,Social Science,0,x,x,None,NaN,NaN,None


FITB questions for comparison.

In [7]:
fitb_sessions = pd.read_parquet( 'fitb_sessions.parquet' )
fitb_sessions.head()

,student_id,question_id,textbook_id,subject,assigned,pattern,first_attempt,second_attempt,second_attempt_elapsed,rating
0,JUURKN3H4CDXWE3FKS7X,000044a0a8292d4f97a4535930bce09145efb5385530eb...,9781003842415,Education,0,+,+,None,NaN,None
1,QGUEFT6V4ETMACKJHZCY,000049b7768af4a214b17ebcf6bfb7a4ab8b1bc19646d8...,9781351263429,Social Science,0,+,+,None,NaN,None
2,ECJZSKV55EX588P73WMX,00005b3c2943ad85c640385240f79445c1a2ab9f12da56...,9781000856255,Language Arts & Disciplines,0,-r,-,None,2.0,None
3,ZCFFE32QY4M3877NPJXD,00006b3a4cbec390d08fb24a29d131a6a1b12ac216dff3...,9781506338118,Education,0,-r+,-,+,1.0,None
4,B6A3W8HDZPVM3GU4DQFR,0000b627c5cf2bfc2df475519eaad44d226145b1a0da8e...,9781071815373,Social Science,0,-+,-,+,25.0,None


## 2. Methods

### 2.2. Data Collection

>This resulted in a dataset of 83,624 LLM-enabled question sessions (56,944 exam question and 26,680 C&C), encompassing 92,719 interaction events, 23,750 questions, 14,696 students, and 1,929 textbooks.

In [8]:
sessions = pd.concat( [ exam_sessions, cc_sessions ] )
print( f'{len( sessions )} sessions' )
print( f'{len( exam_sessions )} exam question sessions' )
print( f'{len( cc_sessions )} C&C sessions' )
print( f'{sessions.pattern.apply( len ).sum()} interactions' )
print( f'{sessions.question_id.nunique()} questions' )
print( f'{sessions.student_id.nunique()} students' )
print( f'{sessions.textbook_id.nunique()} textbooks' )

83624 sessions
56944 exam question sessions
26680 C&C sessions
92719 interactions
23750 questions
14696 students
1929 textbooks


>For comparative purposes, data from the standard FITB questions were retrieved for the same textbooks and timeframe, resulting in 1,142,891 sessions spanning 236,511 questions.

In [9]:
sessions = fitb_sessions
print( f'{len( sessions )} sessions' )
print( f'{sessions.question_id.nunique()} questions' )

1142891 sessions
236511 questions


### 2.3. Analysis

#### 2.3.2. Feedback Usage

In the dataset, student answer attempts are classified using shorthand symbols to represent their accuracy and authenticity. Although these symbols (+, -, x) are not used in the paper, they correspond directly to the categories used, defined in the following table:

| Category    | Symbol | Description                                                                                             |
| ----------- | ------ | ------------------------------------------------------------------------------------------------------- |
| Correct     | `+`    | The response accurately addressed the key distinction between terms.                                    |
| Incorrect   | `-`    | The response did not sufficiently answer the question, despite appearing to be a genuine effort.        |
| Non-Genuine | `x`    | The response did not constitute a legitimate attempt (e.g., random characters, “idk”, irrelevant text). |

## 3. Results and Discussion

### 3.1. Performance Metrics

#### 3.1.1. Engagement

>**Table 1**<br/>
Number of students answering per question by type and assignment context. Assigned questions include mean and quartiles; unassigned questions include only the mean.

In [10]:
exam_sessions[ 'question_type' ] = 'Exam'
cc_sessions[ 'question_type' ] = 'C&C'
fitb_sessions[ 'question_type' ] = 'FITB'
sessions = pd.concat( [ exam_sessions, cc_sessions, fitb_sessions ] )

In [11]:
print( 'Unassigned:' )
sessions[ sessions.assigned == 0 ].groupby( 'question_type' ).apply(
    lambda g: g.groupby( 'question_id' ).student_id.nunique().mean().round( 1 ), include_groups=False
)

Unassigned:


question_type
C&C     2.8
Exam    2.4
FITB    3.9
dtype: float64

In [12]:
print( 'Assigned:' )
sessions[ sessions.assigned == 1 ].groupby( 'question_type' ).apply(
    lambda g: g.groupby( 'question_id' ).student_id.nunique().describe().round( 1 ), include_groups=False
)[ [ 'mean', '25%', '50%', '75%' ] ]

Assigned:


student_id,mean,25%,50%,75%
question_type,,,,
C&C,55.5,3.8,28.0,90.5
Exam,51.7,4.0,21.0,62.0
FITB,84.5,24.0,60.0,143.0


>A Mann–Whitney U test confirmed that significantly more LLM-enabled questions were answered in assigned contexts than in unassigned contexts (U = 1.23 × 10<sup>6</sup>, _p_ < .001).

In [13]:
sessions = pd.concat( [ exam_sessions, cc_sessions ] )
students_per_questions = [ g.groupby( 'question_id' ).student_id.nunique() for _, g in sessions.groupby( 'assigned' ) ]
mannwhitneyu( *students_per_questions )

MannwhitneyuResult(statistic=np.float64(1233914.0), pvalue=np.float64(5.028858079333808e-209))

#### 3.1.2. Difficulty and Persistence

>**Table 2**<br/>
Difficulty and persistence rates by question type and assignment context. Difficulty is defined as the percentage of first attempts marked correct; persistence is the percentage of initially incorrect attempts that were ultimately followed by a correct one.

In [14]:
sessions = pd.concat( [ cc_sessions, fitb_sessions ] )
sessions.groupby( [ 'assigned', 'question_type' ] ).first_attempt.agg(
    difficulty=lambda fa: ( fa == '+' ).mean().round( 3 ) * 100
)

difficulty
assigned question_type            
0        C&C                  50.6
         FITB                 65.9
1        C&C                  59.8
         FITB                 79.9

In [15]:
sessions[ sessions.first_attempt != '+' ].groupby( [ 'assigned', 'question_type' ] ).pattern.agg(
    persistence=lambda p: p.str.contains( '\+' ).mean().round( 3 ) * 100
)

persistence
assigned question_type             
0        C&C                    8.2
         FITB                  61.2
1        C&C                   16.9
         FITB                  94.7

>Specifically, a chi-square test showed that the proportion of correct first attempts for C&C was significantly higher in the assigned context compared to unassigned (χ² = 207.87, _p_ < .001).

In [16]:
sessions = cc_sessions
contingency_data = sessions.groupby( 'assigned' ).first_attempt.agg(
    correct=lambda fa: ( fa == '+' ).sum(),
    incorrect=lambda fa: ( fa != '+' ).sum()
)
contingency_data = contingency_data.stack().reset_index()
contingency_data.columns = 'context result count'.split()
contingency_data.context = contingency_data.context.map( { 0: 'unassigned', 1: 'assigned' } )
contingency_table = contingency_data.pivot( index='context', columns='result', values='count' )
display( contingency_table )
chi2_contingency( contingency_table )

result,correct,incorrect
context,,
assigned,5711,3836
unassigned,8675,8458


Chi2ContingencyResult(statistic=np.float64(207.86773149588936), pvalue=np.float64(4.0093817946069676e-47), dof=1, expected_freq=array([[5147.79392804, 4399.20607196],
       [9238.20607196, 7894.79392804]]))

>For C&C questions, a chi-square test indicated that persistence was significantly higher in assigned contexts (χ² = 204.21, _p_ < .001).

In [17]:
contingency_data = sessions[ sessions.first_attempt != '+' ].groupby( 'assigned' ).pattern.agg(
    persist=lambda p: p.str.contains( '\+' ).sum(),
    non_persist=lambda p: ( ~p.str.contains( '\+' ) ).sum()
)
contingency_data = contingency_data.stack().reset_index()
contingency_data.columns = 'context result count'.split()
contingency_data.context = contingency_data.context.map( { 0: 'unassigned', 1: 'assigned' } )
contingency_table = contingency_data.pivot( index='context', columns='result', values='count' )
display( contingency_table )
chi2_contingency( contingency_table )

result,non_persist,persist
context,,
assigned,3189,647
unassigned,7766,692


Chi2ContingencyResult(statistic=np.float64(204.21222534525572), pvalue=np.float64(2.5158030276399398e-46), dof=1, expected_freq=array([[3418.20237514,  417.79762486],
       [7536.79762486,  921.20237514]]))

#### 3.1.3. Non-Genuine Responses

>Non-genuine responses are lower for students in the assigned group for the open-ended questions: exam questions 11.8% assigned versus 16.8% unassigned and C&C questions 15.2% assigned compared to 19.1% unassigned.

>The FITB questions have 6.6% non-genuine responses for assigned versus 3.9% unassigned.

In [18]:
sessions = pd.concat( [ exam_sessions, cc_sessions, fitb_sessions ] )
sessions.groupby( [ 'assigned', 'question_type' ] ).first_attempt.agg(
    non_genuine=lambda fa: fa.value_counts( normalize=True ).loc[ 'x' ].round( 3 ) * 100
)

non_genuine
assigned question_type             
0        C&C                   19.1
         Exam                  16.8
         FITB                   3.9
1        C&C                   15.2
         Exam                  11.8
         FITB                   6.6

>Chi-square tests confirm these differences are statistically significant for both exam questions (χ² = 200.01, _p_ < .001) and C&C questions (χ² = 63.86, _p_ < .001).

In [19]:
for sessions in [ exam_sessions, cc_sessions ]:
    question_type = sessions.question_type.iloc[ 0 ]
    print( f'{question_type}:' )
    contingency_data = sessions.groupby( 'assigned' ).first_attempt.agg(
        genuine=lambda fa: ( fa != 'x' ).sum(),
        non_genuine=lambda fa: ( fa == 'x' ).sum()
    )
    contingency_data = contingency_data.stack().reset_index()
    contingency_data.columns = 'context result count'.split()
    contingency_data.context = contingency_data.context.map( { 0: 'unassigned', 1: 'assigned' } )
    contingency_table = contingency_data.pivot(index='context', columns='result', values='count')
    display( contingency_table )
    print( chi2_contingency(contingency_table) )
    print()

Exam:


result,genuine,non_genuine
context,,
assigned,12444,1663
unassigned,35655,7182


Chi2ContingencyResult(statistic=np.float64(200.00984642660575), pvalue=np.float64(2.0781801260599678e-45), dof=1, expected_freq=array([[11915.78731736,  2191.21268264],
       [36183.21268264,  6653.78731736]]))

C&C:


result,genuine,non_genuine
context,,
assigned,8100,1447
unassigned,13868,3265


Chi2ContingencyResult(statistic=np.float64(63.862414003256845), pvalue=np.float64(1.334189732353263e-15), dof=1, expected_freq=array([[ 7860.88815592,  1686.11184408],
       [14107.11184408,  3025.88815592]]))



#### 3.1.4. Student Ratings

>**Table 3**<br/> 
Thumbs up and thumbs down ratings per 1,000 student-question sessions, by question type and assignment context.

In [20]:
sessions = pd.concat( [ exam_sessions, cc_sessions, fitb_sessions ] )
ratings = sessions.groupby( [ 'question_type', 'assigned' ] ).rating.value_counts( normalize=True, dropna=False ).round( 5 ).to_frame() * 1000
ratings.columns = [ 'rating' ]
ratings = ratings.reorder_levels( [ 'question_type', 'rating', 'assigned' ] ).sort_index( ascending=[ True, False, True ] )
ratings[ ratings.index.get_level_values( 'rating' ).notna() ]

rating
question_type rating      assigned        
C&C           thumbs_up   0           2.22
                          1           0.21
              thumbs_down 0           0.93
                          1           0.42
Exam          thumbs_up   0           3.67
                          1           0.57
              thumbs_down 0           2.08
                          1           0.64
FITB          thumbs_up   0           2.30
                          1           0.06
              thumbs_down 0           1.45
                          1           0.09

### 3.2. Feedback Usage

>The analysis focuses on cases in which the first attempt was incorrect (C&C 28.4%) or non-genuine (exam question 15.5%, C&C 17.7%).

In [21]:
print( f"{cc_sessions.pattern.str.startswith( '-' ).mean():.1%} incorrect C&C" )
print( f"{exam_sessions.pattern.str.startswith( 'x' ).mean():.1%} non-genuine exam" )
print( f"{cc_sessions.pattern.str.startswith( 'x' ).mean():.1%} non-genuine C&C" )

28.4% incorrect C&C
15.5% non-genuine exam
17.7% non-genuine C&C


>...only 18.2% of exam-question sessions and 13.2% of C&C sessions with a non-correct first attempt proceeded to a second attempt.

In [22]:
print( f"{exam_sessions[ exam_sessions.first_attempt == 'x' ].second_attempt.notna().mean():.1%} second attempt exam" )
print( f"{cc_sessions[ cc_sessions.first_attempt != '+' ].second_attempt.notna().mean():.1%} second attempt C&C" )

18.2% second attempt exam
13.2% second attempt C&C


>**Table 4**<br/>
Time interval (s) between first and second student attempts, by question type, answer pattern (e.g., incorrect → correct), and assignment context.

In [23]:
exam_sessions[ exam_sessions.first_attempt == 'x' ].groupby(
    [ 'assigned', 'first_attempt', 'second_attempt' ]
).second_attempt_elapsed.describe()[ [ '25%', '50%', '75%' ] ]

25%   50%    75%
assigned first_attempt second_attempt                    
0        x             +               15.00  24.0   56.0
                       x               16.25  40.5  114.0
1        x             +               11.00  14.0   27.0
                       x                9.00  21.0   52.0

In [24]:
cc_sessions[ cc_sessions.first_attempt != '+' ].groupby(
    [ 'assigned', 'first_attempt', 'second_attempt' ]
).second_attempt_elapsed.describe()[ [ '25%', '50%', '75%' ] ]

25%   50%     75%
assigned first_attempt second_attempt                     
0        -             +               39.00  73.0  125.50
                       -               51.00  79.0  144.50
                       x               15.00  58.0  162.00
         x             +               17.00  26.5   53.00
                       -               50.00  77.0  112.00
                       x               11.00  18.5   44.25
1        -             +               18.00  29.0   59.00
                       -               36.75  49.0   83.50
                       x               24.50  55.0  115.00
         x             +               12.00  15.0   22.00
                       -               28.00  60.0   91.00
                       x                7.00  11.0   21.50

>**Table 5**<br/>
Token-level textual overlap (percentage) between initial LLM-generated feedback and student second attempt, by question type, answer pattern (e.g., incorrect → correct), and assignment context.

In [25]:
def preprocess_text( text ):
    """
    Lowercases text, removes punctuation but keeps letters/digits, and normalizes spacing.
    Returns a cleaned string suitable for token-level comparison.
    """
    text = text.lower()
    # Remove punctuation/special characters (but keep letters a-z, digits 0-9, and whitespace)
    text = re.sub( r'[^a-z0-9\s]', '', text )
    # Normalize multiple spaces/tabs/newlines into a single space
    text = ' '.join( text.split() )

    return text

def token_based_difflib_ratio( a, b ):
    """
    Returns a float in [0.0, 1.0] indicating how similar two texts are,
    based on token-level difflib (order-sensitive).
    """
    a_tokens = preprocess_text( a ).split()
    b_tokens = preprocess_text( b ).split()
    # Create a SequenceMatcher on the token lists
    matcher = difflib.SequenceMatcher( None, a_tokens, b_tokens )

    return matcher.ratio()

In [26]:
for ( events, sessions ) in [ ( exam_events, exam_sessions ), ( cc_events, cc_sessions ) ]:
    for ( question_id, student_id ), session_events in events.groupby( [ 'question_id', 'student_id' ] ):
        # Only want multi-attempt sessions
        if len( session_events ) == 1:
            continue
        e1 = session_events.iloc[ 0 ]
        e2 = session_events.iloc[ 1 ]
        # Compute overlap between first answer's feedback and second answer
        similarity = token_based_difflib_ratio( e1.feedback, e2.answer )
        sessions.loc[ ( sessions.question_id == question_id ) & ( sessions.student_id == student_id ), 'second_attempt_overlap' ] = similarity

In [27]:
sessions = exam_sessions
sessions[ sessions.first_attempt == 'x' ].groupby(
    [ 'assigned', 'first_attempt', 'second_attempt' ]
).second_attempt_overlap.describe().round( 3 )[ [ '25%', '50%', '75%' ] ] * 100

25%   50%   75%
assigned first_attempt second_attempt                  
0        x             +               28.0  79.2  84.5
                       x                0.0   3.9   9.2
1        x             +               59.6  77.6  83.1
                       x                0.0   4.5  10.4

In [28]:
sessions = cc_sessions
sessions[ sessions.first_attempt != '+' ].groupby(
    [ 'assigned', 'first_attempt', 'second_attempt' ]
).second_attempt_overlap.describe().round( 3 )[ [ '25%', '50%', '75%' ] ] * 100

25%   50%   75%
assigned first_attempt second_attempt                  
0        -             +               25.1  39.4  78.7
                       -               16.0  23.3  29.2
                       x                0.0   0.0  13.8
         x             +               69.9  85.2  95.7
                       -               15.4  20.3  29.1
                       x                0.0   0.0   8.0
1        -             +               35.4  63.9  84.8
                       -               15.7  19.3  22.5
                       x                4.2  12.0  20.4
         x             +               71.2  81.8  95.1
                       -               18.8  28.1  38.0
                       x                0.0   0.0   5.1